$$\newcommand{\ket}[1]{\left|{#1}\right\rangle}$$
$$\newcommand{\bra}[1]{\left\langle{#1}\right|}$$

# Quantum Magnetism, Criticality, and the Variational Ansatz

In the previous notebook we ran a VQE on a Hamiltonian that was handed to us, and we
checked the answer by diagonalising a $4\times4$ matrix. That is enough to see the
machinery work, but it says nothing about *why* one would want a variational algorithm,
or what makes one ansatz better than another.

Here we take a model from many-body physics: the **transverse-field Ising model** (TFIM).
It is the standard minimal example of a quantum phase transition, it is realised in
Rydberg-atom arrays and superconducting simulators, and, crucially for us, it is exactly
solvable. So we will always know the right answer.

The plan:

1. Build the model and look at its two limits.
2. Find its $\mathbb{Z}_2$ symmetry, and see why a finite quantum system never actually
   breaks it.
3. Solve it exactly by mapping **spins to fermions** with a Jordan--Wigner
   transformation. This gives us the free-fermion dispersion, the ground-state energy,
   and the gap that closes at the critical point.
4. Build an ansatz out of the Hamiltonian itself, and find how deep it has to be.
5. Run a VQE across the phase transition and watch what happens when the circuit is too
   shallow.
6. Measure the entanglement entropy and extract the central charge of the underlying
   conformal field theory.

Nothing needs to be downloaded: the model is defined by two numbers.

> **Exercises.** Places to write your own code are marked `YOUR CODE HERE`. Each is
> followed by a worked solution in a **collapsed cell**, which appears as a thin
> clickable bar. Click it to expand the code when you want to compare, and run it as
> normal. Try the exercise first.

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.optimize import OptimizeWarning
from scipy.sparse.linalg import eigsh

# qiskit-algorithms passes a legacy option through to SciPy's L-BFGS-B. Harmless,
# but it prints a warning on every call, which makes the output below hard to read.
warnings.filterwarnings("ignore", category=OptimizeWarning)

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector, partial_trace, entropy
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B

backend = AerSimulator()

---
# 1. The transverse-field Ising model

Take $N$ spin-$\tfrac12$ degrees of freedom on a ring. Neighbouring spins want to align
along $z$, and an external magnetic field along $x$ wants to point every spin along $x$.
Those two demands are incompatible, because $\hat{Z}$ and $\hat{X}$ do not commute, and
the entire physics of the model comes from that conflict:

\begin{equation}
    \hat{H}(g) = -J\left[\sum_{j=0}^{N-1} \hat{Z}_j \hat{Z}_{j+1}
                 + g\sum_{j=0}^{N-1} \hat{X}_j\right],
    \qquad \hat{Z}_N \equiv \hat{Z}_0 .
\end{equation}

$J>0$ is the ferromagnetic coupling and $g = h/J$ the dimensionless field strength. We
set $J=1$ throughout, so $g$ is the only parameter. Look at the two limits:

- **$g = 0$.** Only the coupling survives. The ground states are $\ket{00\cdots0}$ and
  $\ket{11\cdots1}$: every spin aligned, and *two* of them, related by flipping all
  spins. This is the **ordered** or ferromagnetic phase.
- **$g \to \infty$.** Only the field survives. The unique ground state is
  $\ket{+}^{\otimes N}$, a product state with no correlations at all. This is the
  **disordered** or paramagnetic phase.

Somewhere in between, the character of the ground state has to change. It does so at
$g_c = 1$, and not smoothly: the change is a genuine **quantum phase transition**,
driven by competition between non-commuting terms rather than by temperature. It happens
at $T = 0$.

Building the Hamiltonian is three lines, which is the point: unlike a molecule, this
Hamiltonian *is* the physical definition of the system.

In [ ]:
def tfim(N, g, J=1.0, pbc=True):
    '''Transverse-field Ising Hamiltonian H = -J[ sum_j Z_j Z_{j+1} + g sum_j X_j ].'''
    terms = []

    bonds = range(N) if pbc else range(N - 1)
    for j in bonds:
        k = (j + 1) % N
        label = ["I"] * N
        label[j] = label[k] = "Z"
        terms.append(("".join(reversed(label)), -J))     # reversed: Qiskit is little-endian

    for j in range(N):
        label = ["I"] * N
        label[j] = "X"
        terms.append(("".join(reversed(label)), -J * g))

    return SparsePauliOp.from_list(terms)


N = 8                      # ring of 8 spins; everything below scales with this
H = tfim(N, g=1.0)
print(f"{N} spins, {len(H)} Pauli terms, Hilbert space dimension {2**N}")
print(H.to_list()[:3], "...")

In [ ]:
# The two limits, checked against exact diagonalisation.
# Note: we use g = 0.1 rather than g = 0 exactly. At g = 0 the two lowest states are
# exactly degenerate, and any numerical eigensolver is then free to return an arbitrary
# vector from that two-dimensional subspace rather than the physical combination.
for g, description in [(0.1, "g = 0.1  (nearly pure Ising)"),
                       (50.0, "g = 50   (nearly pure field)")]:
    w, v = np.linalg.eigh(tfim(N, g).to_matrix())
    weights = np.abs(v[:, 0]) ** 2
    print(f"{description}   E0 = {w[0]:9.4f}   E1 - E0 = {w[1]-w[0]:.3e}")
    for i in np.argsort(weights)[::-1][:2]:
        print(f"      |{format(i, f'0{N}b')}>  weight {weights[i]:.4f}")
    print(f"      all weights lie in [{weights.min():.5f}, {weights.max():.5f}];"
          f"  a uniform superposition would give {1/2**N:.5f}")

Near $g=0$ the two lowest states are almost exactly degenerate, and the ground state is
split evenly between the two aligned configurations. At large $g$ the ground state is unique and spread evenly
over *all* $2^N$ basis states, which is what $\ket{+}^{\otimes N}$ looks like in the
computational basis. The degeneracy is the fingerprint of the ordered phase, and it is
the first thing we should understand.

---
# 2. $\mathbb{Z}_2$ symmetry and why finite systems do not break it

Flipping every spin, $\hat{Z}_j \to -\hat{Z}_j$ and $\hat{X}_j \to \hat{X}_j$, leaves
$\hat{H}$ alone: the coupling term contains $\hat{Z}$ in pairs and the field term does
not contain $\hat{Z}$ at all. The operator that implements this is

\begin{equation}
    \hat{P} = \prod_{j=0}^{N-1}\hat{X}_j ,
\end{equation}

with $\hat{P}^2 = \hat{I}$, so its eigenvalues are $\pm1$ and the symmetry group is
$\mathbb{Z}_2$.

### Exercise 1

Verify that $[\hat{H},\hat{P}] = 0$ for a few values of $g$, and find the eigenvalue of
$\hat{P}$ in the ground state and the first excited state.

Hints: `SparsePauliOp("X"*N)` builds $\hat{P}$; `A @ B - B @ A` then `.simplify()` gives
the commutator, and `np.allclose(op.coeffs, 0)` tests whether it vanishes. For the
eigenvalue, `np.linalg.eigh` returns eigenvectors as *columns*, so `v[:, 0]` is the
ground state.

In [ ]:
P = SparsePauliOp("X" * N)

# YOUR CODE HERE
#
# for g in (0.3, 1.0, 2.0):
#     H = tfim(N, g)
#     ...

In [ ]:
# Solution
P_matrix = P.to_matrix()

for g in (0.3, 1.0, 2.0):
    H = tfim(N, g)
    commutator = (H @ P - P @ H).simplify(atol=1e-10)
    w, v = np.linalg.eigh(H.to_matrix())

    parity = lambda psi: np.real(psi.conj() @ P_matrix @ psi)
    print(f"g = {g:4.1f}   [H, P] = 0: {np.allclose(commutator.coeffs, 0)}"
          f"    <P> ground = {parity(v[:, 0]):+.4f}"
          f"    <P> first excited = {parity(v[:, 1]):+.4f}")

The ground state always has $\hat{P} = +1$ and the first excited state always has
$\hat{P} = -1$, at every $g$. Since these are exact eigenstates of a finite Hermitian
matrix, the ground state is *symmetric*, never symmetry-broken. Yet we said the $g=0$
ground states were $\ket{00\cdots0}$ and $\ket{11\cdots1}$, neither of which is
symmetric.

Both statements are true. The finite-$N$ eigenstates are the symmetric and antisymmetric
combinations,

\begin{equation}
    \ket{\pm} = \frac{1}{\sqrt{2}}\left(\ket{00\cdots0} \pm \ket{11\cdots1}\right),
\end{equation}

which are GHZ (cat) states. Spontaneous symmetry breaking only happens in the
thermodynamic limit, where the splitting between the two closes and any infinitesimal
perturbation selects one of the aligned states. At finite $N$ the splitting is small but
non-zero, and the following exercise shows *how* small.

### Exercise 2

Compute $E_1 - E_0$ at $g = 0.5$ (ordered phase) for $N = 4, 6, 8, 10$ and plot it on a
log scale. Do the same at $g = 2$ (disordered phase). What is the qualitative difference?

In [ ]:
# YOUR CODE HERE

In [ ]:
# Solution
sizes = [4, 6, 8, 10]
splittings = {}
for g in (0.5, 2.0):
    splittings[g] = []
    for n in sizes:
        w = np.linalg.eigvalsh(tfim(n, g).to_matrix())
        splittings[g].append(w[1] - w[0])
    print(f"g = {g}:  " + "  ".join(f"N={n}: {s:.2e}" for n, s in zip(sizes, splittings[g])))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(sizes, splittings[0.5], "o-", label="$g = 0.5$ (ordered)")
ax.semilogy(sizes, splittings[2.0], "s-", label="$g = 2.0$ (disordered)")
ax.set_xlabel("$N$")
ax.set_ylabel("$E_1 - E_0$")
ax.set_title("Splitting of the lowest doublet")
ax.legend()
plt.show()

In the ordered phase the splitting falls off exponentially with $N$: the two cat states
become degenerate, and in the thermodynamic limit the symmetry breaks. In the disordered
phase the gap is $O(1)$ and independent of size: the ground state is unique and stays
unique.

This is worth holding on to when you interpret VQE results. If your optimiser returns a
state at $g < 1$, it has no reason to pick $\ket{00\cdots0}$ over
$\ket{11\cdots1}$, and the true eigenstate is the cat superposition of both. A long-range
entangled state, in other words, which is exactly the kind of thing a shallow circuit
will struggle with. Section 5 makes that concrete.

---
# 3. Exact solution: from spins to free fermions

The TFIM is one of the few interacting quantum models that can be solved in closed form,
and the tool that does it is the **Jordan--Wigner transformation**. Note the direction of
travel here. In quantum chemistry, Jordan--Wigner is used to turn fermions into qubits so
that a quantum computer can hold them. Here we use it the other way: we turn spins into
fermions, because the resulting fermions turn out to be *free*, and free fermions are
easy.

## Setting up

First a change of basis. Apply a Hadamard to every site, which exchanges
$\hat{X} \leftrightarrow \hat{Z}$:

\begin{equation}
    \hat{H}' = -J\left[\sum_j \hat{X}_j\hat{X}_{j+1} + g\sum_j \hat{Z}_j\right].
\end{equation}

This is a unitary transformation, so the spectrum is untouched. We do it because the
Jordan--Wigner string is built from $\hat{Z}$, and in this form the $\hat{Z}$ term becomes
a simple number operator.

Now define fermionic operators, exactly as in the chemistry case:

\begin{equation}
    \hat{c}_j = \left(\bigotimes_{l<j}\hat{Z}_l\right)\otimes
    \frac{\hat{X}_j + i\hat{Y}_j}{2}.
\end{equation}

The occupation is $\hat{n}_j = \hat{c}^\dagger_j\hat{c}_j = (\hat{I}-\hat{Z}_j)/2$, so
$\ket{1}$ is an occupied mode and $\ket{0}$ an empty one. Two consequences follow
immediately:

\begin{equation}
    \hat{Z}_j = \hat{I} - 2\hat{n}_j, \qquad
    \hat{X}_j\hat{X}_{j+1} = (\hat{c}^\dagger_j - \hat{c}_j)(\hat{c}^\dagger_{j+1} + \hat{c}_{j+1}).
\end{equation}

The second one is the miracle. Naively $\hat{X}_j$ alone is horrible under
Jordan--Wigner, since it carries a string of $\hat{Z}$s all the way back to the origin.
But in the *product* of two neighbours the strings overlap and cancel everywhere except
on sites $j$ and $j+1$. What is left is **quadratic** in fermion operators, and a
quadratic Hamiltonian is a free theory.

In [ ]:
def jw_annihilation(p, n_qubits):
    '''Fermionic annihilation operator c_p as a SparsePauliOp, under Jordan-Wigner.'''
    def label(local_op):
        chars = ["I"] * n_qubits
        for q in range(p):          # the Z string on every mode below p
            chars[q] = "Z"
        chars[p] = local_op
        return "".join(reversed(chars))

    return SparsePauliOp.from_list([(label("X"), 0.5), (label("Y"), 0.5j)])


c_ops = [jw_annihilation(p, N) for p in range(N)]
cd_ops = [op.adjoint() for op in c_ops]

print("c_0 =", c_ops[0].to_list())
print("c_3 =", c_ops[3].to_list(), "  <- note the Z string")

### Exercise 3

Check the claim. Build the quadratic fermionic Hamiltonian with **open** boundaries,

\begin{equation}
    \hat{H}'_{\rm OBC} = -J\sum_{j=0}^{N-2}(\hat{c}^\dagger_j - \hat{c}_j)(\hat{c}^\dagger_{j+1} + \hat{c}_{j+1})
    - Jg\sum_{j=0}^{N-1}\left(\hat{I} - 2\hat{c}^\dagger_j\hat{c}_j\right),
\end{equation}

and verify that it equals the rotated spin Hamiltonian
$-J[\sum_j \hat{X}_j\hat{X}_{j+1} + g\sum_j\hat{Z}_j]$ on the open chain.

Open boundaries first, because the ring has a subtlety we will come to next.

In [ ]:
def rotated_tfim(N, g, J=1.0, pbc=True):
    '''The Hadamard-rotated model: -J[ sum_j X_j X_{j+1} + g sum_j Z_j ].'''
    terms = []
    for j in (range(N) if pbc else range(N - 1)):
        k = (j + 1) % N
        label = ["I"] * N
        label[j] = label[k] = "X"
        terms.append(("".join(reversed(label)), -J))
    for j in range(N):
        label = ["I"] * N
        label[j] = "Z"
        terms.append(("".join(reversed(label)), -J * g))
    return SparsePauliOp.from_list(terms).simplify()


# YOUR CODE HERE

In [ ]:
# Solution
g_test = 0.7
identity = SparsePauliOp("I" * N)

hopping = [-1.0 * ((cd_ops[j] - c_ops[j]) @ (cd_ops[j + 1] + c_ops[j + 1]))
           for j in range(N - 1)]
field = [-g_test * (identity - 2 * (cd_ops[j] @ c_ops[j])) for j in range(N)]

H_fermion = sum(hopping + field[1:], field[0]).simplify(atol=1e-10)
H_spin = rotated_tfim(N, g_test, pbc=False)

print("open chain, quadratic fermion form == rotated spin model:",
      np.allclose(H_fermion.to_matrix(), H_spin.to_matrix()))

## The boundary term

On a ring there is one extra bond, from site $N-1$ back to site $0$, and its string does
*not* cancel. Working it through gives

\begin{equation}
    \hat{X}_{N-1}\hat{X}_0 = -\,(\hat{c}^\dagger_{N-1} - \hat{c}_{N-1})
                             (\hat{c}^\dagger_0 + \hat{c}_0)\,\hat{P},
    \qquad \hat{P} = \prod_j \hat{Z}_j = \prod_j (\hat{I} - 2\hat{n}_j),
\end{equation}

where $\hat{P}$ is the fermion parity, which is precisely the $\mathbb{Z}_2$ symmetry
operator from section 2 (in the rotated basis). So the model is only *strictly* free
within a sector of fixed parity, and the two sectors see different boundary conditions
for the fermions:

- even parity ($\hat{P}=+1$): **antiperiodic**, momenta $k = \dfrac{(2n+1)\pi}{N}$;
- odd parity ($\hat{P}=-1$): **periodic**, momenta $k = \dfrac{2\pi n}{N}$.

We saw in section 2 that the ground state always sits in the even sector, so the
antiperiodic momenta are the ones we need.

In [ ]:
# The ring, with the parity factor on the closing bond
parity_op = SparsePauliOp("Z" * N)
boundary = 1.0 * ((cd_ops[N - 1] - c_ops[N - 1]) @ (cd_ops[0] + c_ops[0]) @ parity_op)

H_fermion_ring = (H_fermion + boundary).simplify(atol=1e-10)
print("ring, with the -P boundary term:",
      np.allclose(H_fermion_ring.to_matrix(), rotated_tfim(N, g_test, pbc=True).to_matrix()))

## Diagonalising, one step at a time

We now have a Hamiltonian that is quadratic in fermion operators, and everything from
here is linear algebra. It is worth going slowly, because these four steps work for
*any* free-fermion model, not just this one.

### Step 1: multiply out the brackets

Expand $(\hat{c}^\dagger_j - \hat{c}_j)(\hat{c}^\dagger_{j+1} + \hat{c}_{j+1})$ and use
$\hat{c}_j\hat{c}^\dagger_{j+1} = -\hat{c}^\dagger_{j+1}\hat{c}_j$ to tidy up. Three
different kinds of term appear:

\begin{align}
    \hat{H}' = &-J\sum_j \left(\hat{c}^\dagger_j \hat{c}_{j+1} + \hat{c}^\dagger_{j+1}\hat{c}_j\right)
    && \text{hopping} \\
    &-J\sum_j \left(\hat{c}^\dagger_j \hat{c}^\dagger_{j+1} + \hat{c}_{j+1}\hat{c}_j\right)
    && \text{pairing} \\
    &+2Jg\sum_j \hat{c}^\dagger_j\hat{c}_j \;-\; JgN .
    && \text{on-site energy}
\end{align}

The hopping and on-site terms are unremarkable: a fermion moves between neighbouring
sites, and it costs energy $2Jg$ to sit on one. The **pairing** term is the interesting
one. It creates and destroys fermions two at a time, so it does not conserve particle
number, only its parity. That parity is the $\mathbb{Z}_2$ symmetry from section 2,
wearing a different hat.

This is why we will need one step more than for an ordinary tight-binding chain. Going to
momentum space will not be enough on its own.

In [ ]:
# Check the expansion (open chain, where there is no parity factor to worry about)
hopping = [-1.0 * ((cd_ops[j] @ c_ops[j + 1]) + (cd_ops[j + 1] @ c_ops[j]))
           for j in range(N - 1)]
pairing = [-1.0 * ((cd_ops[j] @ cd_ops[j + 1]) + (c_ops[j + 1] @ c_ops[j]))
           for j in range(N - 1)]
on_site = [2 * g_test * (cd_ops[j] @ c_ops[j]) for j in range(N)]

H_expanded = (sum(hopping + pairing + on_site[1:], on_site[0])
              - g_test * N * identity).simplify(atol=1e-10)

print("hopping + pairing + on-site reproduces the quadratic form:",
      np.allclose(H_expanded.to_matrix(), H_fermion.to_matrix()))

### Step 2: go to momentum space

Every term links site $j$ to site $j+1$ in the same way, so the chain is translation
invariant and momentum is conserved. Substituting

\begin{equation}
    \hat{c}_j = \frac{1}{\sqrt{N}}\sum_k e^{ikj}\,\hat{c}_k
\end{equation}

turns each sum over sites into a sum over momenta:

- the hopping term becomes $-2J\cos k \;\hat{c}^\dagger_k \hat{c}_k$,
- the on-site term becomes $+2Jg\;\hat{c}^\dagger_k \hat{c}_k$,
- the pairing term couples $k$ to $-k$, and to nothing else.

That last point is the crucial one. A general interacting Hamiltonian would couple every
momentum to every other; here each $k$ talks only to its mirror image $-k$. So the single
$2^N$-dimensional problem falls apart into $N/2$ small, independent problems, one for each
pair $(k,-k)$. This is what translation invariance buys you.

### Step 3: two levels per momentum pair

Fix one pair $(k,-k)$. Between them the two modes can hold four states:

$$\ket{\text{empty}}, \qquad \hat{c}^\dagger_k\ket{\text{empty}}, \qquad
\hat{c}^\dagger_{-k}\ket{\text{empty}}, \qquad
\hat{c}^\dagger_k\hat{c}^\dagger_{-k}\ket{\text{empty}} .$$

The two singly-occupied states are left alone: the pairing term changes occupation by two,
so it cannot connect them to anything. That leaves the empty state and the doubly occupied
one, which *do* mix. So each momentum pair is nothing more than a **two-level problem**,
and diagonalising a $2\times2$ matrix is all that remains.

The rotation that diagonalises it is the **Bogoliubov transformation**. Its effect is easy
to describe even if the algebra is fiddly: the new quasiparticle operators
$\hat{\eta}_k$ are superpositions of *a particle at $k$* and *a hole at $-k$*, which is
exactly the mixing the pairing term forced on us. The same structure shows up in the BCS
theory of superconductivity, for the same reason.

Carrying it through gives the quasiparticle energies

\begin{equation}
    \boxed{\;\varepsilon_k = 2J\sqrt{1 + g^2 - 2g\cos k}\;}
\end{equation}

We will take this as given rather than grind through it: the algebra is standard but not
especially illuminating, and it is written out carefully in Pfeuty's original paper
(*Ann. Phys.* **57**, 79 (1970)) and in chapter 4 of Sachdev's *Quantum Phase
Transitions*. What matters for us is the *shape* of this function, which we come to in a
moment, and the fact that we can check it numerically, which we do below.

### Step 4: fill the ground state

Each pair $(k,-k)$ settles independently into the lower of its two levels, contributing
$-\varepsilon_k$. There are $N/2$ such pairs, one for each $k>0$, so

\begin{equation}
    E_0 = -\sum_{k>0}\varepsilon_k = -\frac{1}{2}\sum_{k}\varepsilon_k ,
\end{equation}

where the second form runs over all $N$ momenta and the $\tfrac12$ undoes the resulting
double counting. (The constant $-JgN$ dropped in step 1 cancels exactly against a shift
in the two-level blocks, because $\sum_k \cos k = 0$ over the antiperiodic momenta.
Nothing is left over.) In the language of quasiparticles,

\begin{equation}
    \hat{H} = \sum_k \varepsilon_k
    \left(\hat{\eta}^\dagger_k\hat{\eta}_k - \tfrac12\right),
\end{equation}

so the ground state is simply the state that no $\hat{\eta}_k$ can be removed from, and
excited states are built by adding quasiparticles one at a time. That is the exact
spectrum, for any $N$ and any $g$, in closed form.

In [ ]:
def exact_energy(N, g, J=1.0):
    '''Exact ground-state energy of the TFIM ring, from the free-fermion solution.'''
    n = np.arange(-N // 2, N // 2)
    k = (2 * n + 1) * np.pi / N              # antiperiodic: even-parity sector
    epsilon = 2 * J * np.sqrt(1 + g**2 - 2 * g * np.cos(k))
    return -0.5 * np.sum(epsilon)


print(f"{'N':>3} {'g':>5} {'diagonalisation':>18} {'free fermions':>16} {'difference':>13}")
for n in (4, 6, 8, 10):
    for g in (0.0, 0.5, 1.0, 2.0):
        ed = np.linalg.eigvalsh(tfim(n, g).to_matrix())[0]
        ff = exact_energy(n, g)
        print(f"{n:>3} {g:>5.1f} {ed:>18.10f} {ff:>16.10f} {abs(ed-ff):>13.1e}")

Agreement to machine precision, at every size and every field strength. We now have a
reference answer that costs no computation at all, which is exactly what we need to judge
a VQE.

## The thermodynamic limit and the critical point

As $N\to\infty$ the momentum sum becomes an integral, giving the energy per site

\begin{equation}
    \frac{E_0}{N} \;\longrightarrow\; -\frac{J}{\pi}\int_0^\pi
    \sqrt{1 + g^2 - 2g\cos k}\;\mathrm{d}k .
\end{equation}

More importantly, the dispersion tells us where the transition is. The cheapest
excitation costs $\min_k \varepsilon_k$, and since $\cos k$ is largest at $k=0$,

\begin{equation}
    \Delta = \varepsilon_{k=0} = 2J\,|1 - g| .
\end{equation}

The gap closes **linearly** at $g_c = 1$ and nowhere else. A closing gap in the
thermodynamic limit is the definition of a critical point: the correlation length
$\xi \sim 1/\Delta$ diverges, and the ground state stops looking like anything you can
build locally.

In [ ]:
def energy_density_thermodynamic(g, J=1.0):
    integrand = lambda k: np.sqrt(1 + g**2 - 2 * g * np.cos(k))
    return -(J / np.pi) * quad(integrand, 0, np.pi)[0]


g_grid = np.linspace(0, 2, 201)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(g_grid, [energy_density_thermodynamic(g) for g in g_grid],
         "-", color="orange", label=r"$N \to \infty$")
for n, style in [(4, ":"), (8, "--")]:
    ax1.plot(g_grid, [exact_energy(n, g) / n for g in g_grid], style,
             color="tab:blue", alpha=0.8, label=f"$N = {n}$")
ax1.axvline(1.0, color="grey", lw=0.8)
ax1.set_xlabel("$g$")
ax1.set_ylabel("$E_0 / N$")
ax1.set_title("Ground-state energy density")
ax1.legend()

for g in (0.5, 0.9, 1.0, 1.5):
    k = np.linspace(-np.pi, np.pi, 300)
    ax2.plot(k, 2 * np.sqrt(1 + g**2 - 2 * g * np.cos(k)), label=f"$g = {g}$")
ax2.set_xlabel("$k$")
ax2.set_ylabel(r"$\varepsilon_k$")
ax2.set_title(r"Quasiparticle dispersion: the gap closes at $g = 1$")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"energy density at criticality : {energy_density_thermodynamic(1.0):.8f}")
print(f"                       -4/pi  : {-4/np.pi:.8f}")

The energy density is smooth through $g=1$; a quantum phase transition of this kind shows
up in a *derivative*, not in the energy itself. The dispersion is where the transition is
visible: at $g=1$ the gap touches zero at $k=0$ and the excitations become gapless and
linearly dispersing, $\varepsilon_k \approx 2J|k|$. That linear dispersion is a
relativistic spectrum, and it is why the critical point is described by a conformal field
theory. We will measure its central charge in section 6.

---
# 4. An ansatz built from the Hamiltonian

Notebook 2 used `efficient_su2`, a *hardware-efficient* ansatz: layers of arbitrary
single-qubit rotations and generic entanglers, chosen because they are cheap on hardware
and not because they have anything to do with the problem. Notebook 1 contrasted these
with *problem-inspired* ansätze. The TFIM gives us a clean example of the second kind.

Split the Hamiltonian into its two non-commuting pieces,

\begin{equation}
    \hat{H}_{ZZ} = -\sum_j \hat{Z}_j\hat{Z}_{j+1}, \qquad
    \hat{H}_{X} = -\sum_j \hat{X}_j ,
\end{equation}

and alternate the evolutions they generate:

\begin{equation}
    \ket{\psi(\boldsymbol{\theta},\boldsymbol{\phi})} =
    \prod_{l=1}^{p} e^{-i\phi_l \hat{H}_X} e^{-i\theta_l \hat{H}_{ZZ}}\;
    \ket{+}^{\otimes N}.
\end{equation}

This is the **Hamiltonian Variational Ansatz** (HVA). The motivation is adiabatic: if you
started in the ground state at $g=\infty$, which is exactly $\ket{+}^{\otimes N}$, and
slowly turned the coupling on, you would end in the ground state you want. Trotterising
that adiabatic path gives precisely this alternating structure, and the variational
parameters let the optimiser find a better schedule than the one you would have guessed.
The same construction with a cost Hamiltonian and a mixer is what QAOA does.

Three things to notice:

- **$2p$ parameters**, independent of $N$. `efficient_su2` on 8 qubits has 32.
- Both $e^{-i\theta \hat{H}_{ZZ}}$ and $e^{-i\phi \hat{H}_X}$ commute with
  $\hat{P} = \prod_j \hat{X}_j$, and $\ket{+}^{\otimes N}$ has $\hat{P}=+1$. So the ansatz
  **cannot leave the correct symmetry sector**, whereas a generic hardware-efficient
  circuit wanders out of it and wastes parameters climbing back.
- $e^{-i\theta \hat{Z}_j\hat{Z}_{j+1}}$ is a native two-qubit rotation, `rzz`.

In [ ]:
def hva(N, p):
    '''Hamiltonian variational ansatz for the TFIM ring, p layers, 2p parameters.'''
    theta = ParameterVector("θ", p)      # ZZ coupling layers
    phi = ParameterVector("φ", p)        # transverse field layers

    qc = QuantumCircuit(N)
    qc.h(range(N))                       # prepare |+>^N, the g -> infinity ground state

    for l in range(p):
        for j in range(N):
            qc.rzz(2 * theta[l], j, (j + 1) % N)      # rzz(a) = exp(-i a ZZ / 2)
        for j in range(N):
            qc.rx(2 * phi[l], j)                      # rx(a)  = exp(-i a X / 2)
    return qc


ansatz = hva(N, p=2)
print(f"N = {N}, p = 2:  {ansatz.num_parameters} parameters, depth {ansatz.decompose().depth()}")
ansatz.draw("mpl", style="iqp", fold=40)

### Exercise 4: how deep does it need to be?

Fix $g = 1$ and increase $p$ until the ansatz reproduces the exact ground-state energy.
Do it for $N = 4$, $6$ and $8$. Is there a pattern?

Three practical notes. Optimise on the statevector rather than through the `Estimator`
for now, so that shot noise does not confuse the picture. Use small random starting values
(say uniform in $[-0.4, 0.4]$) with a few restarts: near $\boldsymbol{\theta} = 0$ the
circuit is close to the identity, which is a sensible place to begin. And switch from
COBYLA to the gradient-based `L_BFGS_B`. This matters more than it sounds: with a noiseless
cost function and a landscape this structured, COBYLA stalls short of the true minimum at
the larger depths and you would wrongly conclude the ansatz was not expressive enough.
Optimiser choice is part of the algorithm, not a detail.

In [ ]:
def energy_statevector(params, circuit, H_matrix):
    '''Exact <psi(params)|H|psi(params)>, no sampling.'''
    psi = np.asarray(Statevector(circuit.assign_parameters(params)).data)
    return float(np.real(psi.conj() @ H_matrix @ psi))


def best_of(circuit, H_matrix, n_params, restarts=3, maxiter=600):
    best = np.inf
    for seed in range(restarts):
        x0 = np.random.default_rng(seed).uniform(-0.4, 0.4, n_params)
        result = L_BFGS_B(maxiter=maxiter).minimize(
            fun=lambda x: energy_statevector(x, circuit, H_matrix), x0=x0)
        best = min(best, result.fun)
    return best


# YOUR CODE HERE

In [ ]:
# Solution
for n in (4, 6, 8):
    H_matrix = tfim(n, 1.0).to_matrix()
    E_exact = exact_energy(n, 1.0)
    line = []
    for p in range(1, n // 2 + 2):
        E = best_of(hva(n, p), H_matrix, 2 * p)
        line.append(f"p={p}: {abs(E - E_exact)/abs(E_exact):.1e}")
    print(f"N = {n:2d}  (N/2 = {n//2})   relative error   " + "   ".join(line))

The error collapses to machine precision at exactly $p = N/2$, and stays there. This is a
known result for the TFIM: $N/2$ layers of the Hamiltonian variational ansatz are enough
to reach the exact ground state on $N$ sites, at any $g$.

The reason is a **light cone**. One layer applies gates only between nearest neighbours,
so it can correlate sites at most one bond apart. After $p$ layers, information has spread
at most $p$ sites in either direction. On a ring of $N$ sites the furthest two sites are
$N/2$ apart, so $p = N/2$ is the first depth at which every pair of spins can possibly
know about each other. Below that, no choice of parameters can produce a state with
correlations across the ring. This is a circuit-level version of the Lieb--Robinson bound.

That is a satisfying answer, but read it the other way round and it is a warning: the
depth you need is set by the correlation length of the state you are chasing. Section 5
shows what happens when you do not have it.

---
# 5. VQE across the phase transition

Now the main experiment. Sweep $g$ from $0$ to $2$, run a VQE at each value, and compare
against the exact answer. We do it twice: once at $p = N/2$, which we know is expressive
enough, and once at $p = 2$, which is not.

Alongside the energy we track two observables that actually describe the physics:

- the **transverse magnetisation** $m_x = \frac{1}{N}\sum_j \langle \hat{X}_j\rangle$,
  which runs from $0$ deep in the ordered phase to $1$ deep in the disordered one;
- the **long-range correlator** $C = \langle \hat{Z}_0 \hat{Z}_{N/2}\rangle$, between the
  two most distant sites on the ring. This is the order parameter: non-zero means
  ferromagnetic order survives across the whole system.

$\hat{Z}_0\hat{Z}_{N/2}$ is diagonal in the computational basis, so we can evaluate it
straight from the measured bitstring probabilities.

### Exercise 5

Run the sweep. Warm-start each point from the previous one, since the ground state changes
smoothly with $g$ and this saves a lot of function evaluations.

In [ ]:
g_values = np.round(np.linspace(0.0, 2.0, 21), 3)

magnetisation_x = sum(SparsePauliOp("I" * (N - 1 - j) + "X" + "I" * j)
                      for j in range(N)).to_matrix() / N


def long_range_correlator(psi, N):
    '''<Z_0 Z_{N/2}>, evaluated from the amplitudes (both operators are diagonal).'''
    basis = np.arange(len(psi))
    z0 = 1 - 2 * ((basis >> 0) & 1)
    zr = 1 - 2 * ((basis >> (N // 2)) & 1)
    return float(np.sum(np.abs(psi) ** 2 * z0 * zr))


# YOUR CODE HERE
#
# for p in (2, N // 2):
#     x = np.full(2 * p, 0.05)          # warm start, updated as g increases
#     for g in g_values:
#         ...

In [ ]:
# Solution.  Takes about half a minute.
results = {}

for p in (2, N // 2):
    circuit = hva(N, p)
    x = np.full(2 * p, 0.05)
    energies, correlators, mx_values = [], [], []

    for g in g_values:
        H_matrix = tfim(N, g).to_matrix()
        result = COBYLA(maxiter=400).minimize(
            fun=lambda z: energy_statevector(z, circuit, H_matrix), x0=x)
        x = result.x                                   # warm start the next value of g

        psi = np.asarray(Statevector(circuit.assign_parameters(x)).data)
        energies.append(result.fun)
        correlators.append(long_range_correlator(psi, N))
        mx_values.append(float(np.real(psi.conj() @ magnetisation_x @ psi)))

    results[p] = dict(energy=np.array(energies),
                      correlator=np.array(correlators),
                      mx=np.array(mx_values))
    print(f"p = {p} done")

# Exact reference curves
E_exact = np.array([exact_energy(N, g) for g in g_values])
exact_correlator, exact_mx = [], []
for g in g_values:
    w, v = np.linalg.eigh(tfim(N, g).to_matrix())
    exact_correlator.append(long_range_correlator(v[:, 0], N))
    exact_mx.append(float(np.real(v[:, 0].conj() @ magnetisation_x @ v[:, 0])))
exact_correlator = np.array(exact_correlator)
exact_mx = np.array(exact_mx)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 11), sharex=True)

for p, colour, marker in [(2, "tab:red", "s"), (N // 2, "tab:blue", "o")]:
    label = f"HVA $p = {p}$" + ("  (= $N/2$)" if p == N // 2 else "  (too shallow)")
    rel_error = np.abs(results[p]["energy"] - E_exact) / np.abs(E_exact)
    axes[0].semilogy(g_values, rel_error, marker + "-", color=colour, ms=4, label=label)
    axes[1].plot(g_values, results[p]["correlator"], marker, color=colour, ms=5,
                 mfc="none", label=label)
    axes[2].plot(g_values, results[p]["mx"], marker, color=colour, ms=5,
                 mfc="none", label=label)

axes[0].set_ylabel("relative energy error")
axes[0].set_title(f"VQE across the phase transition, $N = {N}$")

axes[1].plot(g_values, exact_correlator, "-", color="orange", label="exact")
axes[1].set_ylabel(r"$\langle Z_0 Z_{N/2}\rangle$   (order parameter)")

axes[2].plot(g_values, exact_mx, "-", color="orange", label="exact")
axes[2].set_ylabel(r"$m_x$")
axes[2].set_xlabel("$g$")

for ax, location in zip(axes, ["upper right", "upper right", "lower right"]):
    ax.axvline(1.0, color="grey", lw=0.8, ls="--")
    ax.legend(loc=location)

plt.tight_layout()
plt.show()

In [ ]:
i_critical = int(np.argmin(np.abs(g_values - 1.0)))
print(f"At g = 1:")
for p in (2, N // 2):
    err = abs(results[p]['energy'][i_critical] - E_exact[i_critical]) / abs(E_exact[i_critical])
    print(f"  p = {p}:  energy error {100*err:6.3f} %"
          f"     <Z_0 Z_{N//2}> = {results[p]['correlator'][i_critical]:.3f}"
          f"   (exact {exact_correlator[i_critical]:.3f})")

## Read the middle panel

At $p = N/2$ everything works: the energy is exact to eight digits, and the order
parameter traces out the phase transition, falling from $1$ deep in the ordered phase to
nearly zero at $g=2$, with the crossover sitting where it should.

At $p = 2$ the story is very different, and the interesting part is the *disagreement
between the panels*. The energy error at $g = 1$ is a few percent, which in most contexts
you would call a decent variational calculation. But the order parameter is wrong by more
than an order of magnitude: the shallow circuit reports essentially zero long-range
correlation at every value of $g$, including deep in the ordered phase where the true
answer is $1$. It has not found a ferromagnet at all. It has found the best product-like
state available within its light cone, and that state happens to have nearly the right
energy because energy is a *local* quantity: it is a sum of nearest-neighbour and
on-site terms, all of which a depth-2 circuit can get right.

This is the lesson worth taking away from the whole tutorial:

> A small energy error does not mean you have the right state. Energy is local and
> forgiving; order parameters and correlations are not. Always validate a VQE against an
> observable that probes the physics you care about.

Notice too that the failure is worst at small $g$ and mild at large $g$, tracking the
correlation length: at $g=2$ correlations decay over about one lattice spacing and a
shallow circuit is fine, while at $g \le 1$ they extend across the ring and it is hopeless.

---
# 6. Entanglement, and the central charge

The light-cone argument says depth has to scale with the range of correlations. There is a
sharper way to say the same thing, in terms of **entanglement entropy**.

Cut the ring in half, trace out one half, and compute the von Neumann entropy of the
reduced state,

\begin{equation}
    S = -\mathrm{Tr}\left(\hat{\rho}_A \ln \hat{\rho}_A\right),
    \qquad \hat{\rho}_A = \mathrm{Tr}_B \ket{\psi}\bra{\psi}.
\end{equation}

$S$ measures how much quantum information crosses the cut, and it is what any method
based on a limited amount of entanglement, tensor networks classically or a fixed-depth
circuit here, ultimately has to pay for.

In [ ]:
def ground_state(N, g):
    '''Ground state via sparse diagonalisation, so larger N stays cheap.'''
    w, v = eigsh(tfim(N, g).to_matrix(sparse=True), k=1, which="SA")
    psi = np.real(v[:, 0])
    return psi / np.linalg.norm(psi)


def half_chain_entropy(psi, N):
    rho_A = partial_trace(Statevector(psi), list(range(N // 2)))
    return float(entropy(rho_A, base=np.e))


entropies = [half_chain_entropy(ground_state(N, g), N) for g in g_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(g_values, entropies, "o-", ms=4)
ax.axhline(np.log(2), color="tab:green", ls=":", label=r"$\ln 2$  (cat state)")
ax.axvline(1.0, color="grey", lw=0.8, ls="--", label="$g_c = 1$")
ax.set_xlabel("$g$")
ax.set_ylabel("$S$  (half-chain, nats)")
ax.set_title(f"Entanglement entropy across the transition, $N = {N}$")
ax.legend()
plt.show()

Two features, both physical:

- **A plateau at exactly $\ln 2$ in the ordered phase.** That is the cat state from
  section 2: $\frac{1}{\sqrt2}(\ket{0\cdots0} + \ket{1\cdots1})$ has exactly one bit of
  entanglement across any cut, no matter where you cut or how large the system is.
- **A peak near the critical point, then decay.** Away from $g_c$ the correlation length
  is finite and $S$ saturates (an *area law*, which in one dimension means a constant).
  At $g_c$ the correlation length diverges and $S$ grows with system size instead. At
  $N=8$ the peak is slightly below $g=1$; it moves towards $g_c$ as $N$ grows.

The growth at criticality is not arbitrary. Conformal field theory predicts

\begin{equation}
    S(N) = \frac{c}{3}\ln\left[\frac{N}{\pi}\sin\frac{\pi \ell}{N}\right] + \text{const},
\end{equation}

for a block of $\ell$ sites in a ring of $N$, where $c$ is the **central charge** of the
CFT describing the critical point. The Ising transition is the $c = 1/2$ free Majorana
fermion. For a half chain, $\ell = N/2$, the sine is $1$, so a plot of $S$ against
$\ln(N/\pi)$ should be a straight line of slope $c/3$.

### Exercise 6

Compute the half-chain entropy at $g = 1$ for $N = 4, 6, 8, 10, 12$, fit the slope, and
extract $c$.

In [ ]:
# YOUR CODE HERE

In [ ]:
# Solution
sizes = np.array([4, 6, 8, 10, 12])
S_critical = np.array([half_chain_entropy(ground_state(int(n), 1.0), int(n)) for n in sizes])

x = np.log(sizes / np.pi)
slope, intercept = np.polyfit(x, S_critical, 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, S_critical, "o", label="numerics")
ax.plot(x, slope * x + intercept, "-", color="orange",
        label=f"fit: slope $= {slope:.4f}$")
ax.set_xlabel(r"$\ln(N/\pi)$")
ax.set_ylabel("$S$  (half-chain, nats)")
ax.set_title("Logarithmic growth of entanglement at criticality")
ax.legend()
plt.show()

print(f"fitted slope    : {slope:.4f}")
print(f"central charge  : c = 3 x slope = {3*slope:.4f}")
print(f"Ising CFT value : c = 0.5")

A handful of spins on a laptop, and we have recovered the central charge of the Ising
conformal field theory to within a percent or so. $c$ is a universal number: it does not
care about $J$, about the lattice, or about which microscopic model you started from,
only about the universality class of the transition. Getting it out of a system of 12
spins is a nice illustration of how quickly universal behaviour sets in.

And it closes the loop on section 4. Entanglement that grows with system size is exactly
what a fixed-depth circuit cannot supply, since a depth-$p$ circuit produces at most
$O(p)$ entanglement across a cut. Simulating a critical point therefore requires depth
growing with $N$, whatever ansatz you use. That is a statement about the physics, not
about Qiskit.

---
# Where to go next

**Put the shots back in.** Everything above used exact statevectors so that we could see
the physics. Swap `energy_statevector` for an `Estimator`-based cost function, as in
notebook 2, and redo the $g$ sweep. The Hamiltonian has $2N$ terms in two commuting
groups ($ZZ$ and $X$), so the measurement cost is modest, but you will find that the
energy differences that distinguish the phases are comparable to the shot noise. Work out
how many shots per point you would need.

**Watch a barren plateau appear.** Sample random parameters for `efficient_su2(n)` and
compute the variance of $\partial C/\partial\theta_1$ for $n = 2,4,6,8,10$. It falls
exponentially. Repeat with the HVA and it does not, because the ansatz has far fewer
parameters and they are structured. This makes the barren-plateau figure from notebook 1
concrete, and it is another argument for problem-inspired ansätze.

**Break the integrability.** Add a longitudinal field, $-h_z\sum_j \hat{Z}_j$, or a
next-nearest-neighbour coupling. The Jordan--Wigner trick fails immediately, since the
model is no longer quadratic in fermions, and there is no exact solution to check against.
This is the honest setting for a variational algorithm, and you can still benchmark
against exact diagonalisation up to $N \approx 20$.

**Change the model.** The XY and Heisenberg chains have the same structure. The
Heisenberg antiferromagnet, $\hat{H} = J\sum_j \vec{\sigma}_j\cdot\vec{\sigma}_{j+1}$, is
a good next target: it is critical for all parameters, it has $SU(2)$ symmetry that a good
ansatz should respect, and its exact energy is known from the Bethe ansatz.

**Time evolution instead of ground states.** The same Trotterised structure that motivated
the HVA can simulate dynamics: prepare a domain wall, evolve it, and watch it spread at
the Lieb--Robinson velocity. That velocity is $2Jg$, readable straight off the dispersion
as $\max_k |\mathrm{d}\varepsilon_k/\mathrm{d}k|$, so you have an exact prediction to
check.